### 自宅近くの走行

In [1]:
# CSVデータの読み込み
import pandas as pd
import folium

df = pd.read_csv("../data/drivedata_at248_20260222/20260222-131217-248-0222.csv")
# 緯度経度の抜き出し、かつ「空行」削除
df = df.loc[:, "latitude":"longitude"].dropna()
# values.tolist()でリスト化します。
path_data = df[['latitude','longitude']].values.tolist()

#地図を作成する
#初めの位置を決める。gps情報の先頭を使う
start_pos = [df.iloc[0]['latitude'], df.iloc[0]['longitude']]
# mapの実態を作る
m = folium.Map(location=start_pos, zoom_start=15)

marker_pos=[
    (path_data[0]),
    (path_data[-1])
    ]

# Polylineを引く
folium.PolyLine(
    locations=path_data, # 走行データ
    color="blue",        # 線の色
    weight=5,            # 線の太さ
    opacity=0.7,          # 線の透明度
    dash_array='10, 5'   # 走行線っぽく
).add_to(m)

# Markerは一つの座標しか受け取れない仕様
for pos in marker_pos:
    folium.Marker(location = pos).add_to(m)

#m.save("248drive_20260222.html")


In [2]:
import os
import folium
import base64
from io import BytesIO
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
from PIL import Image, ImageOps  # ImageOpsを追加

#１．画像から緯度経度を抽出する補助関数
def get_gps_data(fname):
    try:
        img = Image.open(fname)
        exif = img._getexif()
        if not exif: return None

        info = {TAGS.get(tag, tag): val for tag, val in exif.items() if tag in TAGS}
        if "GPSInfo" not in info: return None

        gps = {GPSTAGS.get(t, t): info["GPSInfo"][t] for t in info["GPSInfo"]}

        def to_deg(v):
            return float(v[0]) + float(v[1])/60 + float(v[2])/3600
        
        lat = to_deg(gps["GPSLatitude"])
        lon = to_deg(gps["GPSLongitude"])
        if gps.get("GPSLatitudeRef") == "S": lat =-lat
        if gps.get("GPSLongitudeRef") == "W": lon = -lon
        return [lat, lon]
    except:
        return None
    

# 2.設定（画像のフォルダパスを指定してください）
photo_dir = "../data/drivedata_at248_20260222/images"
#m = folium.Map(location = [35.6, 139.7], zoom_start = 15) #初期位置は後で調整
all_positions = [] # ズーム調整用に座標を貯めるリスト

# 3.フォルダ内の画像をループ処理
for file_name in os.listdir(photo_dir):
    if file_name.lower().endswith((".jpg", ".jpeg")):
        path = os.path.join(photo_dir, file_name)
        pos = get_gps_data(path)

        if pos:
            all_positions.append(pos) # 座標を記録

            # --- リサイズ処理 ---
            img = Image.open(path)
            img = ImageOps.exif_transpose(img)
            img.thumbnail((400, 400)) # 長辺を400pxに縮小（アスペクト比維持）

            # メモリ上で変換（一度ファイルに保存せずにBase64化）
            buffered = BytesIO()
            img.save(buffered, format="JPEG", quality=75) # 画質も少し落として軽量化
            encoded = base64.b64encode(buffered.getvalue()).decode()
            # --- ポップアップ作成 ---
            html = f'<b>{file_name}</b><br><img src="data:image/jpeg;base64,{encoded}" width="200">'
            iframe = folium.IFrame(html, width=250, height=200)

            folium.Marker(
                location=pos,
                popup=folium.Popup(iframe),
                icon=folium.Icon(color='orange', icon='camera')
            ).add_to(m)
# --- 仕上げ：自動ズーム合わせ ---
if all_positions:
    # 全ての座標が収まるように表示範囲を自動調整
    m.fit_bounds(all_positions)

#m.save("248drive_20260222_wt_photo.html")
print(f"完了！{len(all_positions)}枚の写真をプロットしました。")           


完了！49枚の写真をプロットしました。


In [ ]:
# tagをとってくる
import overpy
import folium

api = overpy.Overpass()

# 検索クエリを作成、指定した位置を中心に半径５００m以内の"way"を探す

lat, lon = path_data[0]
query = f"""
(
  way(around:500, {lat}, {lon})["highway"];
);
(._; >;);
out;
"""
result = api.query(query)

# 3. 地図にOSMのタグを表示してみる
#m = folium.Map(location=[lat, lon], zoom_start=16)

for way in result.ways:
    # OSMのタグ(名前、道路種別、車線数)を取得
    name = way.tags.get("name", "名前なし")
    highway = way.tags.get("highway", "不明")
    lanes = way.tags.get("lanes", "不明")

    try:

        #道の形をリスト化
        nodes = [[float(node.lat), float(node.lon)] for node in way.get_nodes()]

        #OSMの道を地図に描画
        folium.PolyLine(
            nodes,
            color="green",
            weight=4,
            opacity=0.6,
            tooltip=f"道路名: {name}<br>種別: {highway}<br>車線数: {lanes}"
        ).add_to(m)
        
    except overpy.exception.DataIncomplete:
        continue # データが欠けている道はスキップ

    

m.save("248drive_20260222_OSM_tag.html")




SyntaxError: invalid syntax (1439058074.py, line 41)